In [47]:
import pandas as pd

import warnings

from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings('ignore')

In [48]:
df = pd.read_csv('honeywell_gold_dataset.csv')
df.shape
df.head()


,timestamp,q[0],q[1],q[2],q[3],delta_q_reset[0],delta_q_reset[1],delta_q_reset[2],delta_q_reset[3],quat_reset_counter,...,v_z_valid,xy_reset_counter,z_reset_counter,vxy_reset_counter,vz_reset_counter,heading_reset_counter,xy_global,z_global,dist_bottom_valid,label
0,0,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,0
1,1,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,0
2,2,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,0
3,3,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,0
4,4,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,0


In [49]:
from sklearn.preprocessing import LabelEncoder
import sklearn as sk

X = df.drop(columns=[
    'label', 'timestamp',
    'lat_x', 'lon_x', 'alt_x',
    'lat_y', 'lon_y', 'alt_y'
])
y = df['label']


le = LabelEncoder()
y = le.fit_transform(y)

cols_to_drop = [col for col in X.columns if X[col].nunique() <= 1]
X = X.drop(columns=cols_to_drop)

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

In [50]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = X.select_dtypes(include=['number']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols)
    ]
)
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [51]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    'KNN' : KNeighborsClassifier(n_neighbors=3),
    'DT' : DecisionTreeClassifier(random_state=42),
    'RF' : RandomForestClassifier(random_state=42),
    'XGBoost' : XGBClassifier(eval_metric='mlogloss',random_state=42),
    'CatBoost' : CatBoostClassifier(verbose=0,random_seed=42),
    'LightGBM' : LGBMClassifier(verbose=-1,random_seed=42),
}

In [52]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'{name}: Recall: {str(recall_score(y_test,y_pred))}, Accuracy: {str(accuracy_score(y_test, y_pred))}, F1: {str(f1_score(y_test, y_pred))}')

KNN: Recall: 0.7645921917278701, Accuracy: 0.71874374874975, F1: 0.7377844088026856
DT: Recall: 0.7135678391959799, Accuracy: 0.7017403480696139, F1: 0.7123287671232876
RF: Recall: 0.7765751836103595, Accuracy: 0.709741948389678, F1: 0.7346864143353447
XGBoost: Recall: 0.8357170467723232, Accuracy: 0.7501500300060012, F1: 0.775883725103176
CatBoost: Recall: 0.8844221105527639, Accuracy: 0.7815563112622524, F1: 0.8073394495412844
LightGBM: Recall: 0.8983378430614611, Accuracy: 0.7889577915583117, F1: 0.8150096440469928


In [53]:
proba_cat = models['CatBoost'].predict_proba(X_test)[:, 1]
proba_lgb = models['LightGBM'].predict_proba(X_test)[:, 1]
proba_xgb = models['XGBoost'].predict_proba(X_test)[:, 1]

avg_proba = (proba_cat + proba_lgb + proba_xgb) / 3.0

y_pred_ensemble = (avg_proba >= 0.5).astype(int)

print("--------------------------------------------------")
print("Ensemble (Soft Voting) - Recall: " + str(recall_score(y_test, y_pred_ensemble)))
print("Ensemble (Soft Voting) - Accuracy: " + str(accuracy_score(y_test, y_pred_ensemble)))
print("Ensemble (Soft Voting) - F1: " + str(f1_score(y_test, y_pred_ensemble)))

--------------------------------------------------
Ensemble (Soft Voting) - Recall: 0.8855817549284886
Ensemble (Soft Voting) - Accuracy: 0.7811562312462492
Ensemble (Soft Voting) - F1: 0.8072586328400282


In [54]:
pred_cat = models['CatBoost'].predict(X_test)
pred_lgb = models['LightGBM'].predict(X_test)
pred_xgb = models['XGBoost'].predict(X_test)

total_votes = pred_cat + pred_lgb + pred_xgb

y_pred_hard_voting = (total_votes >= 2).astype(int)

print("--------------------------------------------------")
print("Ensemble (Hard Voting) - Recall: " + str(recall_score(y_test, y_pred_hard_voting)))
print("Ensemble (Hard Voting) - Accuracy: " + str(accuracy_score(y_test, y_pred_hard_voting)))
print("Ensemble (Hard Voting) - F1: " + str(f1_score(y_test, y_pred_hard_voting)))

--------------------------------------------------
Ensemble (Hard Voting) - Recall: 0.8882875918051798
Ensemble (Hard Voting) - Accuracy: 0.7841568313662732
Ensemble (Hard Voting) - F1: 0.8098678414096916


In [55]:
xgb_model = models['XGBoost']
importances = pd.Series(xgb_model.feature_importances_, index=preprocessor.get_feature_names_out())

print(importances.sort_values(ascending=False).head(10))

num__alt_ellipsoid_x    0.447108
num__c_variance_rad     0.124853
num__vel_n_m_s          0.100986
num__dist_bottom        0.083466
num__alt_ellipsoid_y    0.027545
num__eph_x              0.011669
num__epv_x              0.010883
num__y                  0.008367
num__q[2]               0.007684
num__vy                 0.007589
dtype: float32


## Test innowacji filtra Kalmana — klasyczny baseline (bez ML)

Klasyczna metoda detekcji spoofingu z teorii filtrów Kalmana, zaadaptowana pod ten dataset.

**Idea**: w PX4 EKF (Extended Kalman Filter) na pokładzie drona już zachodzi cały proces fuzji IMU + barometr + GPS. Wyjścia EKF to `lat_x, lon_x, alt_x`. Surowe GPS to `lat_y, lon_y, alt_y`. EKF "ufa" GPS-owi tylko jeśli nowa miara jest spójna z wewnętrzną prognozą — inaczej odrzuca pomiar (innovation gating).

**Statystyka**: dla każdej próbki obliczamy znormalizowaną odległość Mahalanobisa między pozycją EKF a pozycją GPS, ważoną przez raportowaną niepewność GPS:

$$\chi^2 = \frac{(\Delta n)^2 + (\Delta e)^2}{\sigma_h^2} + \frac{(\Delta d)^2}{\sigma_v^2}$$

gdzie $\Delta n, \Delta e, \Delta d$ to różnice pozycji EKF vs GPS w metrach (północ, wschód, dół), a $\sigma_h, \sigma_v$ to GPS-owe `eph_y, epv_y` (1σ horyzontalnej i pionowej niepewności pomiaru).

Pod hipotezą zerową (GPS niespoofowany, niepewność dobrze raportowana) ta statystyka ma rozkład $\chi^2$ z 3 stopniami swobody (oczekiwana wartość ≈ 3). Spoofing wymusza systematyczny rozjazd EKF↔GPS → $\chi^2$ rośnie.

**Czemu to "filtr Kalmana"** — to jest dokładnie test używany przez sam EKF do innovation gating (odrzucania niespójnych pomiarów). Nie trenujemy nic, nie potrzebujemy etykiet — czysta klasyczna statystyka. Ortogonalna do modeli ML powyżej, które dropowały kolumny pozycji.

In [56]:
import numpy as np

def kalman_innovation_chi2(df):
    """Klasyczny test chi-squared innowacji EKF/GPS dla detekcji spoofingu.

    Wejście: surowy DataFrame (przed jakimkolwiek dropowaniem kolumn).
    Wyjście: tablica long o długości len(df) z chi-squared score per próbkę.

    Wzór:
        delta_n = (lat_x - lat_y/1e7) * 111000              [m]
        delta_e = (lon_x - lon_y/1e7) * 111000 * cos(lat)   [m]
        delta_d = alt_x - alt_y/1000                        [m]
        chi2 = (delta_n^2 + delta_e^2) / eph_y^2 + delta_d^2 / epv_y^2
    """
    lat_y_deg = df['lat_y'] / 1e7
    lon_y_deg = df['lon_y'] / 1e7
    alt_y_m   = df['alt_y'] / 1000.0

    cos_lat = np.cos(np.radians(df['lat_x']))
    delta_n = (df['lat_x'] - lat_y_deg) * 111_000.0           # różnica EKF-GPS, oś N (m)
    delta_e = (df['lon_x'] - lon_y_deg) * 111_000.0 * cos_lat # różnica EKF-GPS, oś E (m)
    delta_d = df['alt_x'] - alt_y_m                            # różnica EKF-GPS, oś D (m)

    sigma_h = df['eph_y'].clip(lower=0.5)   # raportowana niepewność horyzontalna GPS [m]
    sigma_v = df['epv_y'].clip(lower=0.5)   # raportowana niepewność pionowa GPS [m]

    chi2 = (delta_n**2 + delta_e**2) / sigma_h**2 + delta_d**2 / sigma_v**2
    return chi2.values, delta_n.values, delta_e.values, delta_d.values


In [57]:
# Obliczenie chi-squared score dla całego datasetu
chi2_all, delta_n, delta_e, delta_d = kalman_innovation_chi2(df)

print(f'Chi-squared score obliczony dla {len(chi2_all)} probek.')
print(f'Percentyle: 50%={np.percentile(chi2_all, 50):8.2f}   '
      f'90%={np.percentile(chi2_all, 90):10.2f}   '
      f'99%={np.percentile(chi2_all, 99):10.2f}   '
      f'max={chi2_all.max():10.2f}')
print()
print('Średnia chi-squared per scenario × label:')
print(pd.DataFrame({
    'label': df['label'],
    'chi2': chi2_all,
}).groupby(['label'])['chi2'].agg(['mean','median','std']).round(2))


Chi-squared score obliczony dla 24992 probek.
Percentyle: 50%=    2.35   90%=   1057.31   99%=   4490.41   max=   5652.25

Średnia chi-squared per scenario × label:
         mean  median     std
label                        
0        1.40    1.32    0.65
1      680.74  500.45  975.13


In [58]:
# Ewaluacja na tym samym splicie co modele ML
import sklearn as sk
from sklearn.metrics import precision_recall_curve, roc_auc_score

idx_full = np.arange(len(df))
y_full = df['label'].values

idx_tr, idx_te, y_tr, y_te = sk.model_selection.train_test_split(
    idx_full, y_full, test_size=0.2, shuffle=True, random_state=42
)

chi2_tr = chi2_all[idx_tr]
chi2_te = chi2_all[idx_te]

# Strojenie progu na zbiorze treningowym
prec, rec, thr = precision_recall_curve(y_tr, chi2_tr)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
bi = int(np.argmax(f1s))
best_thr = float(thr[bi-1]) if 0 < bi <= len(thr) else float(np.median(chi2_tr))

print(f'Wybrany prog chi-squared = {best_thr:.3f}  (z PR-curve na trainingu)')
print(f'(dla porownania: pod hipoteza zerowa chi^2 z 3 stopniami swobody ma 95% kwantyl ~7.81)')
print()

pred_kf = (chi2_te >= best_thr).astype(int)

print('Test innowacji Kalmana (chi-squared, klasyczny baseline bez ML):')
print(f'  Recall:   {recall_score(y_te, pred_kf):.4f}')
print(f'  Accuracy: {accuracy_score(y_te, pred_kf):.4f}')
print(f'  F1:       {f1_score(y_te, pred_kf):.4f}')
print(f'  ROC AUC:  {roc_auc_score(y_te, chi2_te):.4f}')
print()
print('Dla porownania (modele ML powyzej, ten sam split):')
print(f'  KNN:                 F1 ~ 0.74')
print(f'  Random Forest:       F1 ~ 0.73')
print(f'  XGBoost:             F1 ~ 0.78')
print(f'  CatBoost / LightGBM: F1 ~ 0.81-0.82')
print(f'  Hard voting:         F1 ~ 0.81')


Wybrany prog chi-squared = 2.766  (z PR-curve na trainingu)
(dla porownania: pod hipoteza zerowa chi^2 z 3 stopniami swobody ma 95% kwantyl ~7.81)

Test innowacji Kalmana (chi-squared, klasyczny baseline bez ML):
  Recall:   0.8616
  Accuracy: 0.9284
  F1:       0.9257
  ROC AUC:  0.9149

Dla porownania (modele ML powyzej, ten sam split):
  KNN:                 F1 ~ 0.74
  Random Forest:       F1 ~ 0.73
  XGBoost:             F1 ~ 0.78
  CatBoost / LightGBM: F1 ~ 0.81-0.82
  Hard voting:         F1 ~ 0.81
